# ZelusBench: Attention Medium

**Mid-tier: all difficulty knobs at moderate levels.**

Medium chains, branching/merging structures, some distractors,
light transforms, 3D. Tests general spatial reasoning competence.

- 15 scenarios, 45 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\\[Query\\s+q_\\d+\\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd))
            for qd, rp in zip(query_dicts, parts)]


SEEDS = 15
print(f"ZelusBench Attention Medium")
print(f"Seeds: {SEEDS} | Total: {SEEDS} scenarios")

In [ ]:
@kbench.task(name="zelusbench_attn_medium")
def zelusbench_attn_medium(llm) -> tuple[float, float]:
    """Combined medium: all knobs at moderate difficulty.
    Returns (mean_accuracy, std_dev)."""

    _start = time.time()
    print(f"Running {SEEDS} medium scenarios...")
    print("=" * 60)

    all_scores = []

    for seed in range(SEEDS):
        rng = random.Random(seed)
        structure = rng.choice([DAGStructure.BRANCHING, DAGStructure.MERGING])
        cfg = ScenarioConfig(
            dim=3,
            min_chain_depth=5, max_chain_depth=8,
            dag_structure=structure,
            distractor_level=DistractorLevel.LOW,
            transform_density=TransformDensity.LIGHT,
            transform_types=["rotation", "translation"],
            query_types=[QueryType.POSITION, QueryType.DISTANCE],
            point_def_types=["cartesian_offset", "magnitude_direction", "magnitude_polar"],
            num_queries=3, num_splits=3,
            seed=seed,
        )
        gen = ScenarioGenerator(cfg)
        scenario = gen.generate(f"medium_s{seed}")

        response = llm.prompt(scenario.prompt)
        scores = score_response(response, scenario)
        all_scores.extend(scores)

        avg = float(np.mean([s.score for s in scores]))
        q_depths = [s.chain_depth for s in scores]
        tiers = [s.tier.name for s in scores]
        print(f"  [{seed+1}/{SEEDS}] dag={structure.name} avg={avg:.2f} q_depths={q_depths} tiers={tiers}")

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    kbench.assertions.assert_true(
        overall >= 0,
        expectation=f"Medium: model should produce valid answers (got {overall:.3f})"
    )

    return overall, std


zelusbench_attn_medium.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_medium